Import libraries


In [ ]:
from src.data.load import load_datasets_from_yaml

Import functions

In [ ]:
datasets = load_datasets_from_yaml("../config/datasets.yaml", nrows=500)

### Load datasets


In [ ]:
basket = pd.read_csv(DATA_PATH / "basket_charged.csv")
added = pd.read_csv(DATA_PATH / "eaf_added_materials.csv")
chemistry = pd.read_csv(DATA_PATH / "eaf_final_chemical_measurements.csv")
gas = pd.read_csv(DATA_PATH / "eaf_gaslance_mat.csv")
temperature = pd.read_csv(DATA_PATH / "eaf_temp.csv")
transformer = pd.read_csv(DATA_PATH / "eaf_transformer.csv")
ferro = pd.read_csv(DATA_PATH / "ferro.csv")
inj = pd.read_csv(DATA_PATH / "inj_mat.csv")
tapping = pd.read_csv(DATA_PATH / "ladle_tapping.csv")
lf_added = pd.read_csv(DATA_PATH / "lf_added_materials.csv")
lf_initial = pd.read_csv(DATA_PATH / "lf_initial_chemical_measurements.csv")

### Dataset sizes


In [ ]:
datasets = {
    "basket": basket,
    "added": added,
    "chemistry": chemistry,
    "gas": gas,
    "temperature": temperature,
    "transformer": transformer,
    "inj": inj
}

for name, df in datasets.items():
    print(name, df.shape)

### Datasets columns

In [ ]:
for name, df in datasets.items():
    print("\n", name)
    print(df.dtypes)

### Datasets head


In [ ]:
for name, df in datasets.items(): 
    print("\n", name)
    display(df.head())

### Datasets variables types

In [ ]:
for name, df in datasets.items():
    print("\n", name)
    print(df.dtypes)

### Varibles type conversion


### Datasets: basket and added

"DATETIME" represents the date and time of the measurement and should be stored as a datetime variable

"MAT_CODE" is a material ID. Even though it consists of numbers, it represents categories rather than quantities, so it should be stored as a string

"CHARGED_AMOUNT" represents a quantity and should be stored as a numeric variable

In [ ]:
cols = basket.select_dtypes(include=["object", "string"]).columns
basket[cols] = basket[cols].astype("string")

basket["MAT_CODE"] = basket["MAT_CODE"].astype("string")
basket["CHARGED_AMOUNT"] = pd.to_numeric(basket["CHARGED_AMOUNT"], errors="coerce")
basket["DATETIME"] = pd.to_datetime(basket["DATETIME"], format="%Y-%m-%d %H:%M:%S")

cols = added.select_dtypes(include=["object", "string"]).columns
added[cols] = added[cols].astype("string")

added["MAT_CODE"] = added["MAT_CODE"].astype("string")
added["CHARGE_AMOUNT"] = pd.to_numeric(added["CHARGE_AMOUNT"], errors="coerce")
added["DATETIME"] = pd.to_datetime(added["DATETIME"], format="%Y-%m-%d %H:%M:%S")
added.rename(columns={"CHARGE_AMOUNT": "CHARGED_AMOUNT"}, inplace=True)

### Dataset: chemistry

"DATETIME" represents the date and time of the measurement and should be stored as a datetime variable

"POSITIONROW" is the measurement identifier. Even though it consists of numbers, it represents a category rather than quantity, so it should be stored as a string

"VAL*" represents the concentration of elements in steel and should therefore be stored as a numeric variable.

In [ ]:
cols = chemistry.select_dtypes(include=["object", "string"]).columns
chemistry[cols] = chemistry[cols].astype("string")

chemistry["DATETIME"] = pd.to_datetime(chemistry["DATETIME"], format="%Y-%m-%d %H:%M:%S")

chemistry["POSITIONROW"] = chemistry["POSITIONROW"].astype("string")

for col in chemistry.columns:
    if col.startswith("VAL"):
        chemistry[col] = chemistry[col].str.replace(",", ".", regex=False)
        chemistry[col] = pd.to_numeric(chemistry[col], errors="coerce")

### Dataset: gas

"REVTIME" represents the date and time of the measurement and should therefore be stored as a datetime variable.

"O2_AMOUNT", "GAS_AMOUNT", "O2_FLOW", and "GAS_FLOW" represent oxygen and gas measurements and should therefore be stored as numeric variables. The gas refers to a fuel gas used to provide thermal energy to the furnace during the scrap melting stage.

In [ ]:
cols = gas.select_dtypes(include=["object", "string"]).columns
gas[cols] = gas[cols].astype("string")

gas["REVTIME"] = pd.to_datetime(gas["REVTIME"].str.replace(",", "."), format="%Y-%m-%d %H:%M:%S.%f")

gas["O2_AMOUNT"] = gas["O2_AMOUNT"].str.replace(",", ".", regex=False)
gas["O2_AMOUNT"] = pd.to_numeric(gas["O2_AMOUNT"], errors="coerce")

gas["O2_FLOW"] = gas["O2_FLOW"].str.replace(",", ".", regex=False)
gas["O2_FLOW"] = pd.to_numeric(gas["O2_FLOW"], errors="coerce")

gas["GAS_AMOUNT"] = gas["GAS_AMOUNT"].str.replace(",", ".", regex=False)
gas["GAS_AMOUNT"] = pd.to_numeric(gas["GAS_AMOUNT"], errors="coerce")

gas["GAS_FLOW"] = gas["GAS_FLOW"].str.replace(",", ".", regex=False)
gas["GAS_FLOW"] = pd.to_numeric(gas["GAS_FLOW"], errors="coerce")

### Dataset: temperature

"DATETIME" represents the date and time of the measurement and should be stored as a datetime variable

In [ ]:
cols = temperature.select_dtypes(include=["object", "string"]).columns
temperature[cols] = temperature[cols].astype("string")

temperature["DATETIME"] = pd.to_datetime(temperature["DATETIME"], format="%Y-%m-%d %H:%M:%S")

### Dataset: transformer

"STARTTIME" represents the date and time of the measurement and should be stored as a datetime variable

"DURATION" represents the amount of time the electric arc was active at a given transformer tap. For analytical purposes, it will be converted to seconds and stored as a numeric variable

"MW" represents the average electric power (in megawatts) during the interval and should therefore be stored as a numeric variable.

In [ ]:
cols = transformer.select_dtypes(include=["object", "string"]).columns
transformer[cols] = transformer[cols].astype("string")

transformer["STARTTIME"] = pd.to_datetime(transformer["STARTTIME"], format="%Y-%m-%d %H:%M:%S")

transformer["DURATION"] = transformer["DURATION"].str.replace(" ", "", regex=False)
transformer["DURATION"] = pd.to_timedelta(
"00:" + transformer["DURATION"], errors="coerce"
).dt.total_seconds()
transformer["MW"] = transformer["MW"].str.replace(" ", "", regex=False)
transformer["MW"] = pd.to_numeric(transformer["MW"], errors="coerce")

### Dataset: inj

"REVTIME" represents the date and time of the measurement and should therefore be stored as a datetime variable.

"INJ_AMOUNT_CARBON" and "INJ_FLOW_CARBON" represent the amount and flow of carbon injected into the furnace, respectively. They should therefore be stored as numeric variables.

In [ ]:
cols = inj.select_dtypes(include=["object", "string"]).columns
inj[cols] = inj[cols].astype("string")

inj["REVTIME"] = pd.to_datetime(inj["REVTIME"].str.replace(",", "."), format="%Y-%m-%d %H:%M:%S.%f")

inj["INJ_AMOUNT_CARBON"] = inj["INJ_AMOUNT_CARBON"].str.replace(",", ".", regex=False)
inj["INJ_AMOUNT_CARBON"] = pd.to_numeric(inj["INJ_AMOUNT_CARBON"], errors="coerce")

inj["INJ_FLOW_CARBON"] = inj["INJ_FLOW_CARBON"].str.replace(",", ".", regex=False)
inj["INJ_FLOW_CARBON"] = pd.to_numeric(inj["INJ_FLOW_CARBON"], errors="coerce")

### Datasets types after conversion

In [ ]:
for name, df in datasets.items():
    print("\n", name)
    print(df.dtypes)

### Datasets basic statistics


In [ ]:
for name, df in datasets.items():
    print(f"\nDataset: {name}")
    display(df.describe())

### Number of heats


In [ ]:
temperature["HEATID"].nunique()

### Initial observations

The dataset is organized around the variable **HEATID**. Multiple datasets capture different aspects of the process, including:

- charged materials
- transformer electrical operation
- oxygen and gas injection
- carbon injection
- temperature measurements
- chemical composition

This structure suggests that the datasets must later be aggregated at the heat level to create a machine learning dataset.


### Saving pre-processed data

In [ ]:
for name, df in datasets.items():
    df.to_csv(f"../data/interim/{name}.csv", index=False)